# Requirements

This notebook assumes Oracle Database is available locally and uses the `chat_history` table for session storage.

It includes the setup needed to run the `Session Memory with Oracle AI Database` example.

In [1]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    user = "system"
    password = "OraclePwd_2025"  # must match ORACLE_PWD from docker run
    dsn = "localhost:1521/FREEPDB1"
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener is not bound to 0.0.0.0 (run fix_oracle_listener())")
                print("    3. Container is not running (check with check_docker_container())")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running:")
                    print("     1. check_docker_container() - verify container is running")
                    print("     2. fix_oracle_listener() - fix listener binding")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

# Connect to Oracle
conn = connect_to_oracle()

Connection attempt 1/3...
✓ Connected successfully!

Oracle AI Database 26ai Free Release 23.26.1.0.0 - Develop, Learn, and Run for Free


In [2]:
# Azure OpenAI environment setup helpers
import getpass
import os

# Function to securely get and set environment variables
def set_env_securely_azure(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [3]:
# https://azure-agent-ai-foundry-resource.openai.azure.com/
# gpt-4o
# gpt-4.1
# gpt-5

set_env_securely_azure("AZURE_OPENAI_ENDPOINT", "Enter your Azure OpenAI endpoint (e.g. https://<resource>.openai.azure.com): ")
set_env_securely_azure("AZURE_OPENAI_DEPLOYMENT", "Enter your Azure OpenAI deployment name (e.g. gpt-4o): ")
print("RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.")

RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.


In [4]:
from agents import Agent, Runner

In [5]:
# Azure companion: configure openai-agents to use Azure OpenAI via DefaultAzureCredential
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from agents import Agent, set_default_openai_client, set_tracing_disabled

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

# Normalize endpoint in case env var includes /openai or deployment path.
raw_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if "/openai" in raw_endpoint.lower():
    raw_endpoint = raw_endpoint[: raw_endpoint.lower().index("/openai")]

AZURE_OPENAI_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]
# Runner uses Responses API; 2024-10-21 commonly fails on /responses in Azure.
env_api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
if not env_api_version or env_api_version == "2024-10-21":
    AZURE_OPENAI_API_VERSION = "2025-03-01-preview"
else:
    AZURE_OPENAI_API_VERSION = env_api_version

azure_agents_client = AsyncAzureOpenAI(
    azure_endpoint=raw_endpoint,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_ad_token_provider=token_provider,
    # Compatibility for openai<2.x credential gate when using only AAD token provider.
    _enforce_credentials=False,
)

# Guard: fail fast if this ever gets replaced with a sync client
if not isinstance(azure_agents_client, AsyncAzureOpenAI):
    raise TypeError(
        "Expected AsyncAzureOpenAI for agent runs. Restart kernel and rerun this cell before Azure runs."
    )

print(type(azure_agents_client))
print(f"Azure endpoint: {raw_endpoint}")
print(f"Azure deployment: {AZURE_OPENAI_MODEL}")
print(f"Azure API version: {AZURE_OPENAI_API_VERSION}")

# Route Agent/Runner calls to Azure OpenAI for the Azure companion cells
set_default_openai_client(azure_agents_client)
set_tracing_disabled(disabled=True)

research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
      academic research.

      Maintain a professional, concise, and scholarly tone appropriate for research discussions.
    """,
)

<class 'openai.lib.azure.AsyncAzureOpenAI'>
Azure endpoint: https://azure-agent-ai-foundry-resource.openai.azure.com
Azure deployment: gpt-4o
Azure API version: 2025-03-01-preview


In [6]:
import asyncio
import nest_asyncio

# Apply nest_asyncio to patch the event loop
nest_asyncio.apply()

In [7]:
import datetime
import uuid

# Create chat_history table in Oracle
with conn.cursor() as cur:
    # Drop table if exists (for development)
    cur.execute("""
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE chat_history';
        EXCEPTION WHEN OTHERS THEN
            IF SQLCODE != -942 THEN RAISE; END IF;
        END;
    """)
    
    # Create chat_history table
    cur.execute("""
        CREATE TABLE chat_history (
            id VARCHAR2(100) PRIMARY KEY,
            thread_id VARCHAR2(100) NOT NULL,
            role VARCHAR2(20) NOT NULL,
            message CLOB NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        TABLESPACE USERS
    """)
    
    # Create index on thread_id and timestamp for efficient retrieval
    cur.execute("""
        CREATE INDEX idx_thread_timestamp 
        ON chat_history(thread_id, timestamp)
        TABLESPACE USERS
    """)
    
    conn.commit()
    print("✅ Table chat_history created successfully with index.")

✅ Table chat_history created successfully with index.


## Session Memory with Oracle AI Database


In [8]:
from typing import List, Optional, Union
from datetime import datetime
import oracledb
import json
import uuid

class OracleSession:
    """Custom Oracle session implementation following the Session protocol"""
    
    def __init__(
        self, 
        session_id: str, 
        connection,
        table_name: str = "chat_history"
    ):
        """
        Initialize Oracle session storage.
        
        Args:
            session_id: Unique identifier for this conversation session
            connection: Active oracledb connection object
            table_name: Name of the Oracle table storing session data
        """
        self.session_id = session_id
        self.conn = connection
        self.table_name = table_name
    
    async def get_items(self, limit: Optional[int] = None) -> List[dict]:
        """Retrieve conversation history for this session"""
        try:
            with self.conn.cursor() as cur:
                if limit:
                    cur.execute(f"""
                        SELECT message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp ASC
                        FETCH FIRST :limit ROWS ONLY
                    """, {'session_id': self.session_id, 'limit': limit})
                else:
                    cur.execute(f"""
                        SELECT message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp ASC
                    """, {'session_id': self.session_id})
                
                rows = cur.fetchall()
                
                items = []
                for row in rows:
                    # Deserialize JSON from CLOB
                    message_clob = row[0]
                    if message_clob:
                        message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                        items.append(json.loads(message_str))
                
                return items
        
        except Exception as e:
            print(f"Error retrieving items: {e}")
            return []
    
    async def add_items(self, items: List[dict]) -> None:
        """Store new items for this session"""
        try:
            with self.conn.cursor() as cur:
                for item in items:
                    item_id = str(uuid.uuid4())
                    
                    # Serialize the entire item as JSON
                    message_json = json.dumps(item)
                    
                    # Extract role if available, otherwise default to 'system'
                    role = item.get('role', 'system')
                    
                    cur.execute(f"""
                        INSERT INTO {self.table_name} (id, thread_id, role, message, timestamp)
                        VALUES (:id, :session_id, :role, :message, CURRENT_TIMESTAMP)
                    """, {
                        'id': item_id,
                        'session_id': self.session_id,
                        'role': role,
                        'message': message_json
                    })
                
                self.conn.commit()
        
        except Exception as e:
            print(f"Error adding items: {e}")
            self.conn.rollback()
    
    async def pop_item(self, limit: Optional[int] = None) -> Optional[Union[dict, List[dict]]]:
        """
        Remove and return the most recent item(s) for this session.
        """
        try:
            with self.conn.cursor() as cur:
                # Pop a single most-recent item
                if not limit or limit <= 1:
                    cur.execute(f"""
                        SELECT id, message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp DESC
                        FETCH FIRST 1 ROW ONLY
                    """, {'session_id': self.session_id})
                    
                    row = cur.fetchone()
                    
                    if row:
                        item_id, message_clob = row
                        message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                        item = json.loads(message_str)
                        
                        # Delete the item
                        cur.execute(f"""
                            DELETE FROM {self.table_name}
                            WHERE id = :id
                        """, {'id': item_id})
                        
                        self.conn.commit()
                        return item
                    
                    return None
                
                # Pop multiple most-recent items
                cur.execute(f"""
                    SELECT id, message
                    FROM {self.table_name}
                    WHERE thread_id = :session_id
                    ORDER BY timestamp DESC
                    FETCH FIRST :limit ROWS ONLY
                """, {'session_id': self.session_id, 'limit': limit})
                
                rows = cur.fetchall()
                
                if not rows:
                    return []
                
                items = []
                ids_to_delete = []
                
                for row in rows:
                    item_id, message_clob = row
                    message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                    items.append(json.loads(message_str))
                    ids_to_delete.append(item_id)
                
                # Delete all items
                for item_id in ids_to_delete:
                    cur.execute(f"""
                        DELETE FROM {self.table_name}
                        WHERE id = :id
                    """, {'id': item_id})
                
                self.conn.commit()
                return items
        
        except Exception as e:
            print(f"Error popping item(s): {e}")
            self.conn.rollback()
            return None if (not limit or limit <= 1) else []
    
    async def clear_session(self) -> None:
        """Clear all items for this session"""
        try:
            with self.conn.cursor() as cur:
                cur.execute(f"""
                    DELETE FROM {self.table_name}
                    WHERE thread_id = :session_id
                """, {'session_id': self.session_id})
                
                self.conn.commit()
                print(f"✅ Session {self.session_id} cleared successfully.")
        
        except Exception as e:
            print(f"Error clearing session: {e}")
            self.conn.rollback()
    
    def close(self) -> None:
        """
        Note: Connection is managed externally, so we don't close it here.
        """
        pass

Basic Example of an Agent with Session Memory


In [9]:
# # Create an agent
# research_agent = Agent(
#     name="Assistant",
#     instructions="Research the topic and return the most relevant information.",
# )

In [10]:
# Azure companion agent with explicit Azure deployment model
research_agent_azure = Agent(
    name="Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="Research the topic and return the most relevant information.",
)

In [11]:
# # Create an Oracle session instance
# session = OracleSession(
#     session_id="conversation_123", 
#     connection=conn,
#     table_name="chat_history"
# )

In [12]:
# Azure companion Oracle session instance
session_azure = OracleSession(
    session_id="conversation_azure_123",
    connection=conn,
    table_name="chat_history"
)

In [13]:
# # First turn
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Hi my name is Richmond, and I am a AI Memory Engineer researching LLMs and Agent Memory",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [14]:
# First turn (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Hi my name is Richmond, and I am a AI Memory Engineer researching LLMs and Agent Memory",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): Hi Richmond! It’s great to meet you. It sounds like you’re working on a fascinating area of research. Large language models (LLMs) and agent memory are quickly becoming critical topics in AI, as memory systems profoundly impact autonomous AI agents’ ability to store, retrieve, and learn from previous experiences. Below is a detailed overview of concepts and ideas relevant to your work, which may guide your exploration:

---

### **Key Concepts in LLM and Agent Memory**
1. **Memory Systems in AI Agents**
   - AI memory systems enable agents to retain information beyond the immediate context. This results in improved long-term coherence, personalization, adaptability, and reasoning.
   - Core challenges:
     - Balancing memory storage capacity and retrieval efficiency.
     - Preventing memory corruption or forgetting critical knowledge.
     - Ensuring scalability without introducing computational bottlenecks.

2. **Memory Architectures**
   - **Episodic Memory**:
  

In [15]:
# # Second turn
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="What is a paper that introduces the attention mechanism in LLMs?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [16]:
# Second turn (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What is a paper that introduces the attention mechanism in LLMs?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): The landmark paper that introduced the **attention mechanism**, which forms the foundation for modern LLMs, is:

### **Title:** *Attention Is All You Need*  
**Authors:** Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin  
**Published:** NeurIPS, 2017  
**Link to paper:** [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

---

### **Key Contributions of the Paper**
The **"Attention Is All You Need"** paper introduces the **Transformer architecture**, which revolutionized the field of natural language processing (NLP) by replacing recurrent neural networks (RNNs) and long short-term memory (LSTM) networks with the self-attention mechanism. The attention mechanism significantly enhances efficiency, scalability, and contextual understanding in language models.

**Key innovations from the paper:**
1. **Self-Attention Mechanism**:
   - Enables models to selectively focus on diffe

In [17]:
# # Third turn, the agent will remember the previous conversation
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Who were the authors of the paper?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [18]:
# Third turn, the agent will remember the previous conversation (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Who were the authors of the paper?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): The authors of the **"Attention Is All You Need"** paper are:

1. **Ashish Vaswani**  
2. **Noam Shazeer**  
3. **Niki Parmar**  
4. **Jakob Uszkoreit**  
5. **Llion Jones**  
6. **Aidan N. Gomez**  
7. **Łukasz Kaiser**  
8. **Illia Polosukhin**  

This groundbreaking paper was published in 2017 as part of the Neural Information Processing Systems (**NeurIPS**) conference and introduced the **Transformer architecture**, which is now the backbone of almost all modern Large Language Models (LLMs), such as GPT, BERT, and others.


In [19]:
# # Fourth turn - continuing the conversation
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="What was the year of publication?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [20]:
# Fourth turn - continuing the conversation (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What was the year of publication?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): The paper **"Attention Is All You Need"** was published in **2017**. It was presented at the **31st Conference on Neural Information Processing Systems (NeurIPS)**, which took place in Long Beach, California, USA.


In [21]:
# # Change the conversation subject and ensure the agent does't remember the previous conversation
# # Specifiying pop without limit will remove the last item in the session
# await session.pop_item(limit=7)

In [22]:
# Change the conversation subject for Azure companion session
await session_azure.pop_item(limit=7)

[{'id': 'msg_00ac02cc8e605fe6006a1a13a2fe3881969999ae7f665e5f2f',
  'content': [{'annotations': [],
    'text': 'The paper **"Attention Is All You Need"** was published in **2017**. It was presented at the **31st Conference on Neural Information Processing Systems (NeurIPS)**, which took place in Long Beach, California, USA.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'content': 'What was the year of publication?', 'role': 'user'},
 {'id': 'msg_00ac02cc8e605fe6006a1a13a0d7648196a7da4302392073fa',
  'content': [{'annotations': [],
    'text': 'The authors of the **"Attention Is All You Need"** paper are:\n\n1. **Ashish Vaswani**  \n2. **Noam Shazeer**  \n3. **Niki Parmar**  \n4. **Jakob Uszkoreit**  \n5. **Llion Jones**  \n6. **Aidan N. Gomez**  \n7. **Łukasz Kaiser**  \n8. **Illia Polosukhin**  \n\nThis groundbreaking paper was published in 2017 as part of the Neural Information Processing Systems (**NeurIPS

In [23]:
# # Fifth turn: The agent should not remember the conversations about the paper
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="What paper have we been talking about?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [24]:
# Fifth turn: The Azure agent should not remember the conversations about the paper
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What paper have we been talking about?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): I haven’t been given detailed context about the specific paper you’re referring to, Richmond. However, since you mentioned that you're an AI Memory Engineer exploring Large Language Models (LLMs) and Agent Memory, there are prominent papers that might align with your research interests. For example:

1. **"Attention Is All You Need"** (Vaswani et al., 2017): This foundational paper introduced the Transformer architecture, which is at the core of modern LLMs and memory handling in AI systems. Transformers revolutionized how sequential data is processed, leveraging self-attention mechanisms.

2. **"Architects of Memory in Dynamic Agents"**: Research related to incorporating long-term memory in AI agents—studies explore how memory modules (episodic memory, working memory) can be used to improve decision-making and autonomous behavior in agents.

3. **"LlamaIndex"** or frameworks involving memory augmentation for LLMs: Incorporating external memory structures to allow LL

In [25]:
# # Because we limited the session to a few items, the agent should still remember our name at the introduction
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Do you still remember my name?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [26]:
# Check whether Azure companion session still remembers the name
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Do you still remember my name?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): Yes, your name is Richmond! You mentioned you're an AI Memory Engineer researching LLMs and Agent Memory. Let me know how I can assist further with your work or questions!


In [27]:
# # Clear the session
# await session.clear_session()

In [28]:
# Clear the Azure companion session
await session_azure.clear_session()

✅ Session conversation_azure_123 cleared successfully.


In [29]:
# # Because we limited the session to 3 items, the agent should still remember our name at the introduction
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Do you still remember my name?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [30]:
# After clearing, Azure companion session should not remember prior details
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Do you still remember my name?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): I don’t retain information from past conversations for privacy and security reasons, so I don’t know your name unless you provide it again during this interaction. Please feel free to share it if you'd like!
